In [13]:
import os
import time
import json
import requests
from typing import Optional

# Configuration
DELAY = 1.0  # Seconds between requests
MAX_RETRIES = 5
LIMIT_PER_REQUEST = 100 # Max allowed by API is usually 100 for this endpoint

FIELDS = [
    "paperId",
    "corpusId",
    "url",
    "title",
    "abstract",
    "venue",
    "publicationVenue",
    "referenceCount",
    "citationCount",
    "influentialCitationCount",
    "isOpenAccess",
    "openAccessPdf",
    "fieldsOfStudy",
    "s2FieldsOfStudy",
    "publicationTypes",
    "publicationDate",
    "journal",
    "citationStyles",
    "authors",
    "year",
]


def safe_get(url, headers=None, params=None, delay=DELAY, retries=MAX_RETRIES):
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=headers or {}, params=params, timeout=30)
            
            if resp.status_code == 429:
                wait = delay * (2 ** attempt)
                print(f"  [429] Rate limited. Sleeping {wait:.1f}s (Attempt {attempt+1}/{retries})")
                time.sleep(wait)
                continue
            
            if resp.status_code != 200:
                print(f"  [HTTP {resp.status_code}] {resp.text}")
                time.sleep(delay)
                continue
            
            return resp
            
        except requests.RequestException as e:
            wait = delay * (2 ** attempt)
            print(f"  [Error] Connection failed: {e}. Retrying in {wait:.1f}s...")
            time.sleep(wait)
            
    print("  [Failed] Exceeded max retries.")
    return None


def search_author_by_name(api_key: str, author_name: str, limit: int = 5):
    """
    Search for authors. requesting 'paperCount' to help identify the right profile.
    """
    url = "https://api.semanticscholar.org/graph/v1/author/search"
    headers = {"x-api-key": api_key}
    # We ask for paperCount and citationCount to help pick the best match
    params = {"query": author_name, "limit": limit, "fields": "name,paperCount,citationCount,hIndex"}
    
    resp = safe_get(url, headers=headers, params=params)
    if not resp:
        return []
    
    data = resp.json()
    return data.get("data", []) if isinstance(data, dict) else data


def get_author_papers(
    api_key: str,
    author_id: str,
    fields: list,
    limit_per_request: int = LIMIT_PER_REQUEST,
    delay: float = DELAY,
):
    """
    Fetch papers for an author using Offset-based pagination.
    """
    headers = {"x-api-key": api_key}
    all_papers = []
    offset = 0

    print(f"Starting paper fetch for Author ID: {author_id}...")

    while True:
        url = f"https://api.semanticscholar.org/graph/v1/author/{author_id}/papers"
        
        # Explicitly using offset/limit
        params = {
            "fields": ",".join(fields),
            "limit": limit_per_request,
            "offset": offset
        }

        resp = safe_get(url, headers=headers, params=params)
        if resp is None:
            break

        data = resp.json()
        
        # Handle cases where 'data' might be missing or None
        papers_batch = data.get("data", [])
        if not papers_batch:
            print("  No more papers found in this batch. Done.")
            break

        # Normalize shape (sometimes API returns wrapper objects)
        clean_batch = []
        for it in papers_batch:
            if isinstance(it, dict) and "paper" in it:
                clean_batch.append(it["paper"])
            else:
                clean_batch.append(it)

        count_new = len(clean_batch)
        all_papers.extend(clean_batch)
        
        print(f"  Fetched {count_new} papers (Total so far: {len(all_papers)})")
        
        # Pagination Logic:
        # Increment offset by the number of papers received. 
        # If we received fewer than requested, we are at the end.
        offset += count_new
        
        if count_new < limit_per_request:
            print("  Reached end of list (batch smaller than limit).")
            break

        time.sleep(delay)

    # Deduplication
    seen = set()
    unique_papers = []
    for p in all_papers:
        # Try different ID fields
        pid = p.get("paperId") or p.get("paper_id")
        # Fallback to title hash if absolutely no ID (rare)
        if not pid:
            pid = str(hash(p.get("title", "")))
            
        if pid in seen:
            continue
        seen.add(pid)
        unique_papers.append(p)

    return unique_papers


def scrape_author_papers_to_corpus(
    api_key: str,
    author_name: str,
    output_path: str,
    fields: Optional[list] = None,
):
    if fields is None:
        fields = FIELDS

    # 1. Search for the author
    authors = search_author_by_name(api_key, author_name, limit=5)
    if not authors:
        print(f"No authors found for name: {author_name}")
        return None

    # 2. Heuristic: Pick the author with the highest paper count to ensure we get the main profile
    # (Or just default to the first one if you prefer)
    best_author = max(authors, key=lambda x: x.get("paperCount", 0))
    
    # Display found options to user (debug info)
    print(f"Found {len(authors)} profiles for '{author_name}'.")
    print(f"Selected: {best_author.get('name')} (ID: {best_author.get('authorId')}) - Papers: {best_author.get('paperCount')}")

    author_id = best_author.get("authorId")

    # 3. Fetch papers
    papers = get_author_papers(api_key, author_id, fields=fields)
    print(f"Retrieval complete. Unique papers: {len(papers)}")

    # 4. Transform to Corpus format (Dictionary keyed by ID)
    corpus = {}
    for p in papers:
        pid = p.get("paperId")
        cid = p.get("corpusId")
        
        # Key priority: CorpusID -> PaperID -> TitleHash
        if cid:
            key = str(cid)
        elif pid:
            key = str(pid)
        else:
            key = str(hash(p.get("title", "")))

        # Ensure essential fields exist even if API returned None
        p["paperId"] = pid
        p["corpusId"] = cid
        
        corpus[key] = p

    # 5. Save
    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(corpus, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(corpus)} papers to: {output_path}")
    return output_path


if __name__ == "__main__":
    # WARNING: Never share your API key publicly.
    API_KEY = "kUTNILOIk5BhPQNXrYif3vQqdxguwE46KM3tWYJ3" 
    
    OUTPUT_PATH = "C://Users//Admin//Downloads//output.json"
    AUTHOR_NAME = "N. D. Duc" # This usually maps to Nguyen Dinh Duc
    
    scrape_author_papers_to_corpus(API_KEY, AUTHOR_NAME, OUTPUT_PATH, fields=FIELDS)

Found 5 profiles for 'N. D. Duc'.
Selected: N. D. Duc (ID: 32491015) - Papers: 213
Starting paper fetch for Author ID: 32491015...
  Fetched 100 papers (Total so far: 100)
  Fetched 100 papers (Total so far: 200)
  Fetched 9 papers (Total so far: 209)
  Reached end of list (batch smaller than limit).
Retrieval complete. Unique papers: 209
Saved 209 papers to: C://Users//Admin//Downloads//output.json
